# model-v1
Generated from `code/business_entity_resolution/src/run_model.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0 lightgbm==4.7.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/metric.py
"""Macro F0.5 exactly as the challenge defines it.

Per S1 entity: F0.5 = 1.25*P*R / (0.25*P + R).
- No true matches (singleton): 1.0 if the prediction is empty, else 0.0.
- True matches but empty prediction, or no correct IDs: 0.0.
Macro average over ALL S1 entities in the evaluation set; an S1 missing from the
predictions counts as an empty prediction.
"""
from collections.abc import Iterable, Mapping

import polars as pl

BETA2 = 0.25  # beta = 0.5


def parse_ids(s: str | None) -> set[str]:
    """'S2-1,S3-2' -> {'S2-1', 'S3-2'}; '' or None -> set()."""
    return {x.strip() for x in s.split(",") if x.strip()} if s else set()


def f05(pred: Iterable[str], truth: Iterable[str]) -> float:
    pred, truth = set(pred), set(truth)
    if not truth:
        return 1.0 if not pred else 0.0
    tp = len(pred & truth)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(truth)
    return (1 + BETA2) * p * r / (BETA2 * p + r)


def per_entity(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]]) -> pl.DataFrame:
    """One row per S1 in `truth`: f05, precision, recall (P/R null when undefined)."""
    rows = []
    for s1, t in truth.items():
        p, t = set(pred.get(s1, ())), set(t)
        tp = len(p & t)
        rows.append({
            "s1": s1,
            "f05": f05(p, t),
            "precision": tp / len(p) if p else None,
            "recall": tp / len(t) if t else None,
            "n_true": len(t),
            "n_pred": len(p),
        })
    return pl.DataFrame(rows, schema={"s1": pl.String, "f05": pl.Float64, "precision": pl.Float64,
                                      "recall": pl.Float64, "n_true": pl.Int64, "n_pred": pl.Int64})


def report(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]],
           country: Mapping[str, str] | None = None) -> pl.DataFrame:
    """Macro F0.5 overall, per country, and for singletons vs non-singletons."""
    df = per_entity(pred, truth).with_columns(
        pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton")).alias("kind"),
        pl.col("s1").replace_strict(country or {}, default="ALL").alias("country"),
    )
    aggs = [
        pl.len().alias("s1"),
        pl.col("f05").mean().alias("macro_f05"),
        pl.col("precision").mean().alias("mean_precision"),
        pl.col("recall").mean().alias("mean_recall"),
    ]
    parts = [df.select(pl.lit("overall").alias("group"), *aggs)]
    if country:
        parts.append(df.group_by("country").agg(aggs).sort("country").select(pl.format("country={}", "country").alias("group"), *[a.meta.output_name() for a in aggs]))
    parts.append(df.group_by("kind").agg(aggs).sort("kind").select(pl.format("kind={}", "kind").alias("group"), *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


In [ ]:
%%writefile /tmp/src/model.py
"""Match model: LightGBM binary classifier on candidate pairs (features feat-v2, table 6).

Feature sets (the France question: French test candidates are found by fewer search methods, median 2 vs 4):
  A  all 86 features;
  B  A without f_n_methods (how many search methods found the pair);
  C  B without the pruner-derived features (f_p_u50, f_p_gap, f_list_rank): the blocking pruner itself uses the
     method count and the per-method ranks, so its score can carry the same bias.
Per-method scores and ranks stay in every set (they are similarities; "not found" is NaN).

Training S1 exclude the 100k train-split S1 the blocking pruner was trained on (their pruner scores and lists
are in-sample); `A_all` keeps them, to measure that effect. Early stopping uses a held-out tenth of the training
S1 (by S1, never by pair); validation S1 are never used for training or early stopping.
"""
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import polars as pl

SEED = 42
PRUNER_DERIVED = ("f_p_u50", "f_p_gap", "f_list_rank")
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, deterministic=True, verbose=-1,
              num_threads=0)
ROUNDS, EARLY_STOP = 3000, 100


def feature_sets(cols: list[str]) -> dict[str, list[str]]:
    f = [c for c in cols if c.startswith("f_")]
    b = [c for c in f if c != "f_n_methods"]
    return {"A": f, "B": b, "C": [c for c in b if c not in PRUNER_DERIVED]}


def early_stop_mask(s1: pl.Series, share: float = 0.1) -> np.ndarray:
    """True for rows whose S1 falls in the held-out share (stable hash of the id, same S1 -> same side)."""
    return ((s1.hash(seed=SEED) % 1000) < int(share * 1000)).to_numpy()


def train(x: pl.DataFrame, y: np.ndarray, feats: list[str], log=print) -> lgb.Booster:
    """x: s1 + feature columns; y: 0/1. Early stopping on a held-out tenth of the S1."""
    hold = early_stop_mask(x["s1"])
    m = x.select(feats).to_numpy()
    dtr = lgb.Dataset(m[~hold], y[~hold], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(m[hold], y[hold], reference=dtr)
    booster = lgb.train(PARAMS, dtr, ROUNDS, valid_sets=[dva], valid_names=["early_stop"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    log(f"    {len(feats)} features, {int((~hold).sum()):,} fit rows, {int(hold.sum()):,} early-stop rows, "
        f"best iteration {booster.best_iteration}, early-stop logloss {booster.best_score['early_stop']['binary_logloss']:.4f}")
    return booster


def predict(booster: lgb.Booster, x: pl.DataFrame, chunk: int = 2_000_000) -> np.ndarray:
    feats = booster.feature_name()
    return np.concatenate([booster.predict(x[lo:lo + chunk].select(feats).to_numpy(), num_iteration=booster.best_iteration)
                           for lo in range(0, x.height, chunk)] or [np.empty(0)])


def importance(booster: lgb.Booster, name: str) -> pl.DataFrame:
    g = booster.feature_importance("gain")
    return (pl.DataFrame({"variant": name, "feature": booster.feature_name(), "gain": g.astype(float)})
            .with_columns((pl.col("gain") / pl.col("gain").sum()).alias("gain_share")).sort("gain", descending=True))


In [ ]:
%%writefile /tmp/src/selection.py
"""Match selection: from scored candidate pairs to the predicted matches per S1, tuned for macro F0.5.

Rules (all tuned on validation, never on test):
- threshold: keep a pair only if its match probability p >= tau;
- one owner: every S2/S3 record matches at most one S1 (EDA: 0 of 7.6M train records match two), so each
  candidate is kept only for the S1 that gives it the highest p;
- margin: keep a pair only if p >= margin * (best p of that S1), which drops weak extra matches next to a strong one.
F0.5 weighs precision twice, so a wrong match costs more than a missed one: the tuned threshold is usually high.

f05_table() is a vectorised version of metric.py (same definition, checked against it in the tests and in
run_model.py): macro F0.5 over ALL evaluated S1, an S1 without predictions scores 1.0 if it has no true match.
"""
from __future__ import annotations

import itertools

import polars as pl

TAUS = [round(0.05 * i, 2) for i in range(2, 20)]  # 0.10 .. 0.95
MARGINS = [0.0, 0.3, 0.5, 0.7]
BETA2 = 0.25


def select(scored: pl.DataFrame, tau: float, margin: float = 0.0, one_owner: bool = True) -> pl.DataFrame:
    """scored: s1, cand, p. Returns the kept (s1, cand, p) rows."""
    x = scored.filter(pl.col("p") >= tau)
    if one_owner:  # ties: the smaller s1 id wins (deterministic)
        x = x.sort(["cand", "p", "s1"], descending=[False, True, False]).unique("cand", keep="first", maintain_order=True)
    if margin > 0:
        x = x.filter(pl.col("p") >= margin * pl.col("p").max().over("s1"))
    return x.select("s1", "cand", "p")


def f05_table(pred: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame) -> pl.DataFrame:
    """Per evaluated S1: n_pred, n_true, tp, precision, recall, f05. pred / truth: s1, cand; s1s: s1 (+ any
    attribute columns, e.g. country), the S1 to evaluate. S1 missing from pred count as empty predictions."""
    keys = s1s.select("s1")
    pred = pred.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    truth = truth.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    tp = pred.join(truth, on=["s1", "cand"], how="semi").group_by("s1").len("tp")
    x = (s1s.join(pred.group_by("s1").len("n_pred"), on="s1", how="left")
         .join(truth.group_by("s1").len("n_true"), on="s1", how="left")
         .join(tp, on="s1", how="left")
         .with_columns(pl.col("n_pred", "n_true", "tp").fill_null(0)))
    prec = pl.when(pl.col("n_pred") > 0).then(pl.col("tp") / pl.col("n_pred"))
    rec = pl.when(pl.col("n_true") > 0).then(pl.col("tp") / pl.col("n_true"))
    f = (pl.when(pl.col("n_true") == 0).then((pl.col("n_pred") == 0).cast(pl.Float64))
         .when(pl.col("tp") == 0).then(0.0)
         .otherwise((1 + BETA2) * prec * rec / (BETA2 * prec + rec)))
    return x.with_columns(prec.alias("precision"), rec.alias("recall"), f.alias("f05"))


def summary(tab: pl.DataFrame, by: str | None = None) -> pl.DataFrame:
    """Macro F0.5, mean precision / recall (over S1 where defined), per group and overall (+ singletons)."""
    aggs = [pl.len().alias("s1"), pl.col("f05").mean().alias("macro_f05"), pl.col("precision").mean().alias("precision"),
            pl.col("recall").mean().alias("recall"), pl.col("n_pred").mean().alias("pred_per_s1")]
    kind = pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton"))
    parts = [tab.select(pl.lit("overall").alias("group"), *aggs),
             tab.with_columns(kind.alias("_k")).group_by("_k").agg(aggs).sort("_k")
             .select(pl.format("kind={}", "_k").alias("group"), *[a.meta.output_name() for a in aggs])]
    if by:
        parts.insert(1, tab.group_by(by).agg(aggs).sort(by).select(pl.format(f"{by}={{}}", by).alias("group"),
                                                                     *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


def tune(scored: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, taus=TAUS, margins=MARGINS,
         owners=(True, False)) -> pl.DataFrame:
    """Macro F0.5 on s1s for every (tau, margin, one_owner). Sorted best first."""
    rows = []
    for tau, margin, owner in itertools.product(taus, margins, owners):
        tab = f05_table(select(scored, tau, margin, owner), truth, s1s)
        rows.append({"tau": tau, "margin": margin, "one_owner": owner, "macro_f05": tab["f05"].mean(),
                     "pred_per_s1": tab["n_pred"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


In [ ]:
%%writefile /tmp/src/submission.py
"""Write matching_results.tsv / candidate_pairs.tsv in the exact challenge format.

One row per S1 (in the given order), tab-separated, comma-joined IDs, no quoting,
no duplicates, empty second column when there is nothing to list.
"""
from collections.abc import Iterable, Mapping
from pathlib import Path

MATCHING_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


def write_id_lists(path: Path, header: tuple[str, str], s1_ids: Iterable[str],
                   lists: Mapping[str, Iterable[str]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s1 in s1_ids:
            ids = list(dict.fromkeys(i for i in lists.get(s1, ()) if not i.startswith("S1-")))
            f.write(f"{s1}\t{','.join(ids)}\n")
            n += 1
    return n


def write_submission(out_dir: Path, s1_ids: list[str], matches: Mapping[str, Iterable[str]],
                     candidates: Mapping[str, Iterable[str]]) -> None:
    for s1, m in matches.items():
        extra = set(m) - set(candidates.get(s1, ()))
        if extra:
            raise ValueError(f"{s1}: matches not in candidates: {sorted(extra)[:5]}")
    write_id_lists(out_dir / "matching_results.tsv", MATCHING_HEADER, s1_ids, matches)
    write_id_lists(out_dir / "candidate_pairs.tsv", CANDIDATE_HEADER, s1_ids, candidates)


In [ ]:
%%writefile /tmp/src/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_model.py', '--input', '/kaggle/input', '--out-dir', '/kaggle/working', '--validator', '/tmp/src/validate_submission.py']

In [ ]:
"""Model + selection, end to end on Kaggle: train, validate, choose, score test, write and validate the outputs.

1. Train LightGBM variants on features-v2 train (see model.py): A (all features), B (no f_n_methods),
   C (B without the pruner-derived features), each on the train S1 the blocking pruner never saw, plus A_all
   (A on all train S1, to measure the pruner-overlap effect).
2. Validation (99,994 S1, never used for training): the selection (threshold, one owner per S2/S3, margin) is
   tuned on the tune half (even S1 id) and reported on the report half (odd id): macro F0.5 overall, per country,
   singletons vs non-singletons, precision, recall. Every validation S1 counts, including those with no
   candidates.
3. France-like stress test on validation: true pairs found by only 1-2 search methods (French test candidates:
   median 2 methods vs 4 in train), mean probability and recall per variant; and how the blocking pruner treated
   them (kept share by method count, from the unpruned validation list).
4. Choice: the most France-robust variant (C, then B, then A) whose report-half F0.5 is within MAX_LOSS of the best.
5. Test: score the test features with the chosen model, select, write output/matching_results.tsv and
   output/candidate_pairs.tsv (candidates = the blocking v3 set), run the organiser validator (--check-ids), and
   the France checks: predicted matches per S1 by country, and where French matches sit in capped lists.

Usage: python src/run_model.py --input DIR --out-dir DIR --validator utils/validate_submission.py
"""
import argparse
import json
import resource
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
from metric import per_entity  # noqa: E402
from model import feature_sets, importance, predict, train  # noqa: E402
from selection import f05_table, select, summary, tune  # noqa: E402
from submission import write_submission  # noqa: E402

SEED = 42
PRUNER_S1 = 100_000  # run_blocking_eval.py: split role == "train" .sample(100_000, seed=42)
MAX_LOSS = 0.005  # prefer a more France-robust variant if it loses less than 0.5 F0.5 points
ROBUST_ORDER = ["C", "B", "A"]
T0 = time.time()


def log(msg: str) -> None:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    print(f"[{time.time() - T0:7.1f}s peak {peak:5.1f} GB] {msg}", flush=True)


def find(root: Path, name: str) -> Path:
    hits = sorted(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def md(df: pl.DataFrame, max_rows: int = 60) -> str:
    df = df.head(max_rows)
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(fmt(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path, help="folder searched for every input file")
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--validator", default=None)
    ap.add_argument("--variants", nargs="*", default=["A", "B", "C", "A_all"])
    ap.add_argument("--skip-test", action="store_true")
    ap.add_argument("--pruner-s1", type=int, default=PRUNER_S1, help="size of the pruner's train sample (100k; smaller only for tests)")
    a = ap.parse_args()
    out = a.out_dir
    out.mkdir(parents=True, exist_ok=True)
    rd = lambda name, cols=None: pl.read_parquet(find(a.input, name), columns=cols)

    # ---------------------------------------------------------------- data
    split = rd("g25_split.parquet")
    trn = split.filter(pl.col("role") == "train")["source1_entity_id"]
    pruner_s1 = trn.sample(min(a.pruner_s1, trn.len()), seed=SEED)
    val_s1 = split.filter(pl.col("role") == "val").select(pl.col("source1_entity_id").alias("s1"))
    country1 = rd("train_source1.parquet", ["entity_id", "country"]).rename({"entity_id": "s1"})
    val_s1 = val_s1.join(country1, on="s1", how="left")
    gt = rd("train_ground_truth.parquet")
    truth_val = (gt.join(val_s1, left_on="source1_entity_id", right_on="s1", how="semi")
                 .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
                 .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != "")))
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1_tune, s1_rep = val_s1.filter(half == 0), val_s1.filter(half == 1)

    ftr = rd("features_train.parquet")
    labels = rd("train_candidates.parquet", ["s1", "cand", "is_match"])
    ftr = ftr.join(labels, on=["s1", "cand"], how="left")
    assert ftr["is_match"].null_count() == 0, "train features without a label row"
    in_pruner = ftr["s1"].is_in(pruner_s1.implode())
    fva = rd("features_val.parquet")
    fva = fva.join(truth_val.with_columns(pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left").with_columns(
        pl.col("is_match").fill_null(False))
    log(f"train {ftr.height:,} pairs ({int(in_pruner.sum()):,} from the pruner's 100k S1); val {fva.height:,} pairs, "
        f"{val_s1.height:,} S1 (tune {s1_tune.height:,} / report {s1_rep.height:,}), {truth_val.height:,} true pairs")
    blocking_recall = fva["is_match"].sum() / truth_val.height
    sets = feature_sets(ftr.columns)

    # ---------------------------------------------------------------- train + validate every variant
    results, imps, scored_val, boosters, chosen_cfg = [], [], {}, {}, {}
    for v in a.variants:
        fs = sets[v.split("_")[0]]
        rows = ftr if v.endswith("_all") else ftr.filter(~in_pruner)
        log(f"variant {v}: training on {rows['s1'].n_unique():,} S1")
        t0 = time.time()
        b = train(rows.select("s1", *fs), rows["is_match"].cast(pl.Int8).to_numpy(), fs, log=log)
        boosters[v] = b
        b.save_model(str(out / f"model_{v}.txt"), num_iteration=b.best_iteration)
        imps.append(importance(b, v))
        sv = fva.select("s1", "cand", "is_match").with_columns(pl.Series("p", predict(b, fva)))
        scored_val[v] = sv
        grid = tune(sv, truth_val, s1_tune)
        best = grid.row(0, named=True)
        chosen_cfg[v] = {k: best[k] for k in ("tau", "margin", "one_owner")}
        grid.head(10).write_csv(out / f"tune_top_{v}.csv")
        rep = f05_table(select(sv, **chosen_cfg[v]), truth_val, s1_rep)
        sm = summary(rep, by="country")
        results.append({"variant": v, "features": len(fs), "train_s1": rows["s1"].n_unique(), "best_iter": b.best_iteration,
                        "train_s": time.time() - t0, **chosen_cfg[v], "tune_f05": best["macro_f05"],
                        **{r["group"]: r["macro_f05"] for r in sm.iter_rows(named=True)},
                        "precision": sm.filter(pl.col("group") == "overall")["precision"].item(),
                        "recall": sm.filter(pl.col("group") == "overall")["recall"].item(),
                        "pred_per_s1": sm.filter(pl.col("group") == "overall")["pred_per_s1"].item()})
        sm.write_csv(out / f"val_report_{v}.csv")
        log(f"variant {v}: tau {best['tau']}, margin {best['margin']}, one owner {best['one_owner']}; "
            f"report-half macro F0.5 {results[-1]['overall']:.4f}")
    res = pl.DataFrame(results)
    res.write_csv(out / "variants.csv")
    pl.concat(imps).write_csv(out / "importance.csv")

    # exact check of the vectorised F0.5 against metric.py on the report half (first variant)
    v0 = a.variants[0]
    pred0 = select(scored_val[v0], **chosen_cfg[v0])
    ids = s1_rep["s1"].to_list()
    pm = {s: [] for s in ids}
    for s, c in pred0.join(s1_rep, on="s1", how="semi").select("s1", "cand").iter_rows():
        pm[s].append(c)
    tm = {s: [] for s in ids}
    for s, c in truth_val.join(s1_rep, on="s1", how="semi").iter_rows():
        tm[s].append(c)
    exact = per_entity(pm, tm)["f05"].mean()
    fast = f05_table(pred0, truth_val, s1_rep)["f05"].mean()
    assert abs(exact - fast) < 1e-9, f"F0.5 mismatch: metric.py {exact} vs vectorised {fast}"
    log(f"F0.5 check vs metric.py: {exact:.6f} == {fast:.6f}")

    # ---------------------------------------------------------------- France-like stress test (validation)
    nm = fva.select("s1", "cand", pl.col("f_n_methods").cast(pl.Int32).alias("n_methods"), "is_match")
    nm = nm.with_columns(pl.when(pl.col("n_methods") >= 3).then(pl.lit("3+")).otherwise(pl.col("n_methods").cast(pl.String)).alias("methods"))
    stress = []
    for v, sv in scored_val.items():
        kept = select(sv, **chosen_cfg[v]).select("s1", "cand", pl.lit(True).alias("kept"))
        x = nm.join(sv.select("s1", "cand", "p"), on=["s1", "cand"]).join(kept, on=["s1", "cand"], how="left").with_columns(
            pl.col("kept").fill_null(False))
        stress.append(x.group_by("methods", "is_match").agg(pl.len().alias("pairs"), pl.col("p").mean().alias("mean_p"),
                                                            pl.col("p").median().alias("median_p"), pl.col("kept").mean().alias("selected_share"))
                      .with_columns(pl.lit(v).alias("variant")))
    stress = pl.concat(stress).sort("is_match", "methods", "variant", descending=[True, False, False])
    stress.write_csv(out / "stress_methods.csv")
    # the pruner: kept share of TRUE pairs by method count, from the unpruned validation starting list (u50)
    vc = rd("val_candidates.parquet", ["scope", "s1", "cand", "rank_name", "rank_name_city", "rank_name_addr",
                                       "rank_reverse", "rank_rare", "p_u50", "is_match"])
    vc = vc.filter((pl.col("scope") == "state") & pl.col("p_u50").is_not_null())
    vc = vc.with_columns(pl.sum_horizontal(*[pl.col(c).is_not_null() for c in vc.columns if c.startswith("rank_")]).alias("n_methods"),
                         pl.col("p_u50").rank("ordinal", descending=True).over("s1").alias("prank"))
    vc = vc.with_columns(((pl.col("p_u50") >= 0.01) & (pl.col("prank") <= 10)).alias("kept"),
                         pl.when(pl.col("n_methods") >= 3).then(pl.lit("3+")).otherwise(pl.col("n_methods").cast(pl.String)).alias("methods"))
    pruner = vc.group_by("methods", "is_match").agg(pl.len().alias("pairs"), pl.col("kept").mean().alias("pruner_kept_share"),
                                                    pl.col("p_u50").median().alias("median_p_u50")).sort("is_match", "methods", descending=[True, False])
    pruner.write_csv(out / "stress_pruner.csv")
    del vc

    # ---------------------------------------------------------------- choice
    best_f = res["overall"].max()
    ok = {r["variant"]: r["overall"] for r in res.iter_rows(named=True) if r["overall"] >= best_f - MAX_LOSS}
    choice = next((v for v in ROBUST_ORDER if v in ok), res.sort("overall", descending=True)["variant"][0])
    log(f"choice: {choice} (best report-half F0.5 {best_f:.4f}; within {MAX_LOSS}: {ok})")
    cfg = {"variant": choice, **chosen_cfg[choice], "val_report_f05": float(res.filter(pl.col("variant") == choice)["overall"].item()),
           "blocking_recall_val": float(blocking_recall), "max_loss_rule": MAX_LOSS, "features": boosters[choice].feature_name()}
    (out / "selection_config.json").write_text(json.dumps(cfg, indent=1))

    report = [
        "# Model + selection (feat-v2, LightGBM)\n",
        f"Validation: {val_s1.height:,} S1, {truth_val.height:,} true pairs; blocking recall of the candidate set "
        f"{blocking_recall:.4f}. Selection tuned on the tune half ({s1_tune.height:,} S1), reported on the report half "
        f"({s1_rep.height:,} S1). All-empty baseline F0.5 = singleton share. Vectorised F0.5 checked against metric.py "
        f"({exact:.6f}).\n",
        "## Variants (report half)\n", md(res) + "\n",
        f"**Choice: {choice}**: the most France-robust variant within {MAX_LOSS * 100:.1f} F0.5 points of the best. "
        f"Selection: tau {cfg['tau']}, margin {cfg['margin']}, one owner {cfg['one_owner']}.\n",
        f"## Chosen variant {choice}: report half by group\n", md(pl.read_csv(out / f"val_report_{choice}.csv")) + "\n",
        "## France-like stress test: pairs by number of search methods that found them\n", md(stress, 40) + "\n",
        "## Blocking pruner: kept share by number of search methods (unpruned validation list)\n", md(pruner) + "\n",
        "## Feature importance (gain share, top 20 per variant)\n",
        md(pl.concat(imps).group_by("variant", maintain_order=True).head(20).select("variant", "feature", "gain_share"), 100) + "\n",
    ]

    # ---------------------------------------------------------------- test
    if not a.skip_test:
        b = boosters[choice]
        tcols = list(dict.fromkeys(["s1", "cand", *b.feature_name(), "f_list_rank", "f_list_size", "f_n_methods"]))
        fte = pl.read_parquet(find(a.input, "features_test.parquet"), columns=tcols)
        p = predict(b, fte)
        s1_all = rd("test_source1.parquet", ["entity_id", "country"]).rename({"entity_id": "s1"})
        st = fte.select("s1", "cand", "f_list_rank", "f_list_size", "f_n_methods").with_columns(pl.Series("p", p)).join(s1_all, on="s1")
        del fte
        matched = select(st.select("s1", "cand", "p"), **chosen_cfg[choice])
        log(f"test: {st.height:,} pairs scored, {matched.height:,} matches")
        cand = rd("test_candidates.parquet", ["s1", "cand", "p_u50"])
        lists = cand.sort(["s1", "p_u50", "cand"], descending=[False, True, False]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
        mlists = matched.sort(["s1", "p"], descending=[False, True]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
        write_submission(out / "output", s1_all["s1"].to_list(), dict(zip(mlists["s1"].to_list(), mlists["cand"].to_list())),
                         dict(zip(lists["s1"].to_list(), lists["cand"].to_list())))
        del cand, lists
        log("wrote output/matching_results.tsv and output/candidate_pairs.tsv")
        val_msgs = None
        if a.validator:
            import importlib.util
            spec = importlib.util.spec_from_file_location("validate_submission", a.validator)
            vs = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(vs)
            with tempfile.TemporaryDirectory() as td:
                for k in (1, 2, 3):
                    rd(f"test_source{k}.parquet", ["entity_id"]).write_csv(Path(td) / f"test_source{k}.tsv", separator="\t")
                errors, warnings = vs.validate(str(out / "output/matching_results.tsv"), str(out / "output/candidate_pairs.tsv"),
                                               td, check_ids=True)
            val_msgs = {"errors": errors, "warnings": warnings}
            log(f"validator: {len(errors)} error(s), {len(warnings)} warning(s)")
        # France checks
        n = s1_all.join(matched.group_by("s1").len("n"), on="s1", how="left").with_columns(pl.col("n").fill_null(0))
        per_country = n.group_by("country").agg(pl.len().alias("s1"), pl.col("n").mean().alias("matches_per_s1"),
                                                (pl.col("n") > 0).mean().alias("share_with_match"),
                                                *[(pl.col("n") == k).mean().alias(f"share_{k}") for k in (0, 1, 2, 3)],
                                                (pl.col("n") >= 4).mean().alias("share_4plus")).sort("country")
        mk = matched.select("s1", "cand", pl.lit(True).alias("m"))
        sm = st.join(mk, on=["s1", "cand"], how="left").with_columns(pl.col("m").fill_null(False))
        capped = sm.filter(pl.col("f_list_size") >= 10)
        cap_pos = capped.filter(pl.col("m")).group_by("country").agg(
            pl.len().alias("matches_in_capped_lists"), pl.col("f_list_rank").mean().alias("mean_position"),
            (pl.col("f_list_rank") <= 3).mean().alias("share_pos_1_3"), (pl.col("f_list_rank") >= 9).mean().alias("share_pos_9_10")).sort("country")
        cap_rate = capped.group_by("country").agg((pl.col("m") & (pl.col("f_list_rank") >= 9)).sum().alias("matches_at_9_10"),
                                                  pl.col("s1").n_unique().alias("capped_s1")).sort("country")
        cap_pos = cap_pos.join(cap_rate, on="country")
        byp = sm.group_by("country").agg(pl.col("p").median().alias("median_p_all"), pl.col("p").quantile(0.9, "nearest").alias("p90_all"),
                                         pl.col("p").filter(pl.col("m")).median().alias("median_p_matched"),
                                         pl.col("f_n_methods").filter(pl.col("m")).mean().alias("methods_of_matches")).sort("country")
        for name, df in (("test_matches_per_country", per_country), ("test_capped_positions", cap_pos), ("test_scores", byp)):
            df.write_csv(out / f"{name}.csv")
        vtext = "not run" if val_msgs is None else f"{len(val_msgs['errors'])} errors, {len(val_msgs['warnings'])} warnings"
        report += ["## Test (no labels)\n", f"Matches: {matched.height:,} for {s1_all.height:,} test S1. Validator: "
                   f"{vtext}.\n",
                   "### Predicted matches per S1 by country\n", md(per_country) + "\n",
                   "### Capped lists (10 candidates): where the predicted matches sit (position by pruner score)\n", md(cap_pos) + "\n",
                   "### Scores by country\n", md(byp) + "\n"]
        (out / "test_check.json").write_text(json.dumps({"validator": val_msgs, "matches": matched.height}, indent=1, default=str))
        if val_msgs and val_msgs["errors"]:
            (out / "model_report.md").write_text("\n".join(report))
            raise SystemExit(f"VALIDATOR ERRORS: {val_msgs['errors'][:3]}")
    (out / "model_report.md").write_text("\n".join(report))
    log("done")


if __name__ == "__main__":
    main()
